# RealPDE Eval on Kaggle (`local_eval.py`)

Editor-first flow: implement variants locally under `submissions/submission_vN/`, `git push`, then here just `git pull` and run. This notebook never writes `submission.py` — it only pulls the repo and calls `local_eval.py --submission submissions/<variant>`.

1. Pull repo → 2. resolve `--data` → 3. smoke-test each variant on `example_data/` → 4. (optional) score on real `test_real` + stage a baseline `model.pth` → 5. pack Codabench zips → 6. SPS symmetry probe (v4 sym vs asym, same run).

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import torch
print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# --- Resolve --data (example_data fallback, real test_real if attached) ---
from pathlib import Path

REPO = Path(REPO_DIR)
example_data = REPO / 'example_data'
# Kaggle dataset root (cf. continue_cno_kaggle.ipynb): {baseline,test,train_real,train_sim}/ underneath
DATA_ROOT = next((c for c in ['/kaggle/input/datasets/nthday/realpde', '/kaggle/input/realpde'] if Path(c).exists()), None)
print(f'DATA_ROOT -> {DATA_ROOT}')
DATA_DIR = example_data  # default: bundled synthetic smoke test
for cand in ([Path(DATA_ROOT)] if DATA_ROOT else []) + [Path('/kaggle/input/realpde')]:
    if (cand / 'test_real').is_dir() and (cand / 'mean_std_real.pt').exists():
        DATA_DIR = cand
        break
print(f'DATA_DIR -> {DATA_DIR}')
print(f'  example_data present: {(example_data / "test_real").is_dir()}')

# --- List submission variants ---
variants = sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir())
print(f'variants: {variants}')

In [ ]:
# --- Smoke test every variant on example_data (CPU, always works) ---
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
    print(f'\n===== {v} (example_data) =====')
    r = subprocess.run([sys.executable, 'local_eval.py', '--submission', f'submissions/{v}',
                        '--data', './example_data'], cwd=REPO)
    print(f'[{v}] exit code: {r.returncode}')

## Real-data check: 30 random trajectories, CNO, no adaptation (GPU)

Stages 30 seeded trajectories from the attached real dataset into `/kaggle/working/real30/` (truncated copies + protocol-exact stats, via `scripts/stage_real30.py`), then scores `submission_v3` + CNO on GPU. Local CPU would take ~hours — here it's minutes. Skips cleanly if no real dataset is attached.

In [ ]:
# --- Stage 30 real trajectories (seeded, reproducible) ---
# Tune these two lines if you want more coverage (GPU-cheap for v3, pricey for v2).
N_FILES, N_FRAMES, SEED = 30, 200, 42
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
DATA_ROOT = next((c for c in ['/kaggle/input/datasets/nthday/realpde', '/kaggle/input/realpde'] if Path(c).exists()), None)
train_real = None
if DATA_ROOT is not None:
    base = Path(DATA_ROOT) / 'train_real'
    if sorted(base.glob('*.h5')):
        train_real = base
    elif sorted(base.rglob('*.h5')):  # nested layout: most common parent wins
        from collections import Counter
        train_real = Counter(p.parent for p in base.rglob('*.h5')).most_common(1)[0][0]
print(f'train_real -> {train_real}')
if train_real is None:
    print('[skip] no real trajectories attached — attach the dataset with train_real/')
else:
    r = subprocess.run([sys.executable, 'scripts/stage_real30.py', '--src', str(train_real),
                        '--dst', '/kaggle/working/real30', '--n-files', str(N_FILES),
                        '--frames', str(N_FRAMES), '--seed', str(SEED)], cwd=REPO)
    print(f'[stage] exit code: {r.returncode}')

In [ ]:
# --- Score a variant + CNO on the staged real-30 set (GPU when available) ---
# Set VARIANT to the submission folder under test (v4 = calibrated, v3 = plain).
VARIANT = 'submission_v4'
MODEL_HINT = 'cno'  # checkpoint filename hint ('fno' for submission_v12)
import shutil, subprocess, sys, torch
from pathlib import Path

REPO = Path(REPO_DIR)
real30 = Path('/kaggle/working/real30')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
if not (real30 / 'test_real').is_dir():
    print('[skip] /kaggle/working/real30 not staged — run the cell above first')
else:
    roots = [Path('/kaggle/input/datasets/nthday/realpde'), Path('/kaggle/input/realpde'), Path('/kaggle/working/checkpoints')]
    ckpts = [p for r in roots if r.exists() for ext in ('*.pth', '*.pt') for p in r.rglob(ext)]
    cands = sorted([p for p in ckpts if MODEL_HINT in p.name.lower()],
                 key=lambda p: (0 if 'sim_real' in p.name.lower() else 1, p.name))
    ckpt = cands[0] if cands else (sorted(ckpts)[0] if ckpts else None)
    print(f'checkpoint: {ckpt}')
    if ckpt is None:
        print('[skip] no checkpoint found')
    else:
        dst = REPO / 'submissions' / VARIANT / 'model.pth'
        shutil.copy(ckpt, dst)
        print(f'[stage] {ckpt.name} -> submissions/{VARIANT}/model.pth')
        r = subprocess.run([sys.executable, 'local_eval.py', '--submission',
                            f'submissions/{VARIANT}', '--data', str(real30),
                            '--device', device], cwd=REPO)
        print(f'[{VARIANT} real30] exit code: {r.returncode}')
        print('Copy the subscores above into submissions/README.md Leaderboard.')

## SPS symmetry probe (v4): symmetric q90 vs asymmetric [q05, q95], one run

v4's band is symmetric (`pred ± q90(|resid|)`). If signed residuals are skewed, an asymmetric band (`pred+q05`, `pred+q95`) buys coverage at the same width. This cell runs v4 once, mirrors its residual table in signed form under the identical `history`/`table_frames` rule, and scores both bands with `local_eval.sps_component_breakdown` — same predictions, same table state, so the delta is pure symmetry effect. A `mirror_err` near zero proves the signed table reproduces v4's abs-q90 half-width step-for-step. Reports per-channel `q95` vs `|q05|`: ratio ≈ 1 means symmetric, asymmetric adds nothing.

In [ ]:
# --- SPS symmetry probe: v4 symmetric vs asymmetric [q05, q95], same run ---
import importlib.util, shutil
from collections import deque

import numpy as np
import torch

PROBE_SUB = 'submission_v4'
probe_sub = REPO / 'submissions' / PROBE_SUB
if not (probe_sub / 'model.pth').exists():
    _ckpt = None
    for _r in [DATA_ROOT, '/kaggle/input/realpde', '/kaggle/working/checkpoints']:
        _r = Path(_r) if _r else None
        if _r is None or not _r.exists():
            continue
        _cands = sorted([p for p in _r.rglob('*.pth') if 'cno' in p.name.lower()],
                        key=lambda p: (0 if 'sim_real' in p.name.lower() else 1, p.name))
        if _cands:
            _ckpt = _cands[0]
            break
    assert _ckpt is not None, 'no CNO checkpoint found in attached data'
    shutil.copy(_ckpt, probe_sub / 'model.pth')
    print(f'staged model.pth <- {_ckpt}')
_spec = importlib.util.spec_from_file_location('v4_probe_sub', probe_sub / 'submission.py')
V4 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(V4)

from local_eval import Normalizer, build_stream, sps_component_breakdown
import scoring as official

probe_data = Path('/kaggle/working/real30')
if not (probe_data / 'test_real').is_dir():
    probe_data = DATA_DIR
print(f'probe data: {probe_data}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = V4.get_ttt_model(str(probe_sub), device)
stream = build_stream(probe_data)
norm = Normalizer(probe_data / 'mean_std_real.pt').to(device)
C, H, T = 2, model.history, model.table_frames

preds, tgts, sym_lo, sym_hi, asym_lo, asym_hi = [], [], [], [], [], []
signed_tbl, q95_hist, q05_hist = deque(), [], []
prev_pred_n = prev_tg_n = None
n_fallback, mirror_err = 0, 0.0
for s in stream:
    inp = s['input'].unsqueeze(0).to(device)
    tgt = s['target'].unsqueeze(0).to(device)
    if s['is_first']:
        model.reset_ttt_state()
        signed_tbl.clear()
        prev_pred_n = prev_tg_n = None
        prev_arg = None
    else:
        # Mirror v4._update_table in signed form BEFORE ttt_step: identical
        # slicing rule, so the pooled set matches v4's abs table element-wise.
        r = (prev_tg_n - prev_pred_n)[..., :-T, :, :, :C]
        signed_tbl.append(r.detach().reshape(-1, C).cpu())
        while len(signed_tbl) > H:
            signed_tbl.popleft()
        prev_arg = prev_tg_n
    xn, yn = norm.preprocess(inp, tgt)
    pred_n, info = model.ttt_step(xn, prev_arg)
    lon = torch.as_tensor(info['lower']).detach()
    hin = torch.as_tensor(info['upper']).detach()
    assert tuple(lon.shape) == tuple(pred_n.shape) == tuple(hin.shape)
    den = lambda a: norm.postprocess_pred(a.to(device)).squeeze(0).cpu().numpy().astype(np.float32)[..., :C]
    preds.append(den(pred_n.detach()))
    tgts.append(tgt.squeeze(0).cpu().numpy().astype(np.float32)[..., :C])
    sym_lo.append(den(lon))
    sym_hi.append(den(hin))
    if len(signed_tbl):
        pool = torch.cat(list(signed_tbl), dim=0)
        q05 = torch.quantile(pool, 0.05, dim=0)
        q95 = torch.quantile(pool, 0.95, dim=0)
        q05_hist.append(q05.numpy())
        q95_hist.append(q95.numpy())
        # Sanity: signed table reproduces v4's abs-q90 half-width (normalized units).
        w_v4 = ((hin - lon) / 2)[..., :C].mean(dim=(0, 1, 2, 3)).cpu()
        mirror_err = max(mirror_err, float((w_v4 - torch.quantile(pool.abs(), 0.90, dim=0)).abs().max()))
        alo, ahi = lon.clone(), hin.clone()
        alo[..., :C] = pred_n.detach()[..., :C] + q05.to(device).view(1, 1, 1, 1, C)
        ahi[..., :C] = pred_n.detach()[..., :C] + q95.to(device).view(1, 1, 1, 1, C)
        asym_lo.append(den(alo))
        asym_hi.append(den(ahi))
    else:
        n_fallback += 1
        asym_lo.append(den(lon))
        asym_hi.append(den(hin))
    prev_pred_n, prev_tg_n = pred_n.detach(), yn.detach()

P = np.stack(preds, 0)
G = np.stack(tgts, 0)
sym = sps_component_breakdown(P, G, C, lower=np.stack(sym_lo, 0), upper=np.stack(sym_hi, 0))
asym = sps_component_breakdown(P, G, C, lower=np.stack(asym_lo, 0), upper=np.stack(asym_hi, 0))
Q95 = np.stack(q95_hist, 0)
Q05 = np.stack(q05_hist, 0)
print(f'steps {len(stream)}, fallback {n_fallback}, table-mirror err {mirror_err:.2e} (must be ~0)')
assert mirror_err < 1e-5, 'signed table diverged from v4 internals'
for i, ch in enumerate(['u', 'v']):
    ratio = float(np.mean(Q95[:, i]) / max(float(np.mean(np.abs(Q05[:, i]))), 1e-12))
    print(f'{ch}: mean q95 {np.mean(Q95[:, i]):.5f} vs mean |q05| {np.mean(np.abs(Q05[:, i])):.5f} '
          f'(ratio {ratio:.3f})')
for name, d in [('sym-q90', sym), ('asym-[q05,q95]', asym)]:
    print(f'{name}: sps {official.score_sps(d["sps_raw"]):.2f}, coverage {d["coverage"]:.4f}, '
          f'mean_nil {d["mean_nil"]:.3f}')
print('verdict: ratio ~= 1 per channel -> symmetric residuals, asymmetric band adds nothing; '
      'ratio >> 1 or << 1 -> skewed, asymmetric wins.')

In [ ]:
# --- Pack Codabench zips (submission.py at root, shared files injected) ---
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
    print(f'\n===== packing {v} =====')
    r = subprocess.run([sys.executable, 'scripts/make_submission_zip.py', v], cwd=REPO)
    print(f'[{v}] pack exit code: {r.returncode}')
print('\nZips:')
!ls -la dist/ 2>/dev/null || echo '(no dist/ yet — pack a variant first)'